# Tiingo API

Direct requests against the Tiingo end-of-day API via `navi`'s `TiingoClient`.

Run the setup cell first. Requires `TINGO_API_KEY` in `../navi/.env`.

In [ ]:
%reload_ext autoreload
%autoreload 2

import asyncio
from lib.clients import TiingoClient
from lib.utils import print_json_vertical

## Verify credentials

In [ ]:
from lib.env import get_tiingo_api_key, get_tiingo_base_url

key = get_tiingo_api_key()
print(f"Loaded Tiingo key ({len(key)} chars)")
print(f"Tiingo base URL ({get_tiingo_base_url()})")

## `/daily/<ticker>` -- ticker metadata

In [ ]:
async def show_raw_meta(ticker="SPY"):
    async with TiingoClient() as client:
        raw = await client._get(f"/daily/{ticker}")
    print_json_vertical(raw)

await show_raw_meta()

In [ ]:
async def show_meta(ticker="SPY"):
    async with TiingoClient() as client:
        meta = await client.get_meta(ticker)
    print(f"{meta.ticker} -- {meta.name}")
    print(f"exchange: {meta.exchange_code}")
    print(f"range:    {meta.start_date} -> {meta.end_date}")

await show_meta()

## `/daily/<ticker>/prices` -- EOD price series

Provide `startDate`/`endDate` (YYYY-MM-DD) for a range. Works for ETFs, mutual funds (e.g. `VFIAX`), and stocks.

In [ ]:
async def show_raw_prices(ticker="SPY", start_date="2024-01-02", end_date="2024-01-10"):
    async with TiingoClient() as client:
        raw = await client._get(
            f"/daily/{ticker}/prices",
            {"startDate": start_date, "endDate": end_date},
        )
    print(f"{len(raw)} rows; first bar:")
    print_json_vertical(raw[0])

await show_raw_prices()

In [ ]:
async def show_prices(ticker="SPY", start_date="2024-01-02", end_date="2024-01-10"):
    async with TiingoClient() as client:
        series = await client.get_prices(ticker, start_date=start_date, end_date=end_date)
    print(f"{series.ticker}: {len(series.prices)} bars")
    print()
    for p in series.prices:
        print(f"{p.date.date()}  close={p.close:>8.2f}  adj_close={p.adj_close:>12.4f}  vol={p.volume}")

await show_prices()

## Full available history

In [ ]:
async def show_full_history(ticker="SPY"):
    async with TiingoClient() as client:
        series = await client.get_prices(ticker, start_date="1900-01-01")
    first, last = series.prices[0], series.prices[-1]
    print(f"{series.ticker}: {len(series.prices)} bars "
          f"({first.date.date()} -> {last.date.date()})")

await show_full_history()